# UR5e VLA bed — Kaggle smoke (free)
Measures what this GPU can do for SmolVLA before any long run: three 10-step timings, the renderer, the evaluator. Inputs: the private dataset **vla-bed-v2**. Settings: Accelerator GPU (T4 x2 or P100), Internet ON. ≈ 15 min.

In [ ]:
import os, subprocess, sys, time, json, pathlib
REPO = "https://github.com/santapong/RoboLLM.git"; BRANCH = "experiment/ur5e-vla-bed"
ROOT = pathlib.Path("/kaggle/working/RoboLLM")
# Kaggle mounts the uploaded zip under /kaggle/input/<slug>/ with or without the zip's top folder; find the manifest.
hits = [p.parent for p in pathlib.Path("/kaggle/input").rglob("manifest.json") if (p.parent / "train").is_dir()]
assert hits, "add the private dataset vla-bed-v2 to this notebook (Add Input); found: " + str(sorted(str(p) for p in pathlib.Path("/kaggle/input").rglob("*"))[:20])
DATA = hits[0]; print("dataset root", DATA)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(ROOT)], check=True)
os.chdir(ROOT)
link = ROOT / "datasets" / "vla-bed" / "v2"; link.parent.mkdir(parents=True, exist_ok=True)
if not link.exists(): link.symlink_to(DATA)          # every default path in the bed now resolves to the uploaded data
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "sim/vla-bed/requirements-record.txt", "mujoco==3.10.0", "pyyaml", "av"], check=True)  # the bed's physics is not in LeRobot's extras
subprocess.run("apt-get install -y -qq libosmesa6 > /dev/null 2>&1 || true", shell=True)   # MuJoCo fallback renderer
os.environ["MUJOCO_GL"] = "egl"; os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "bf16 native", torch.cuda.is_bf16_supported())
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Which renderer works here? EGL (NVIDIA) first, OSMesa second.
import os, subprocess, sys
def try_gl(backend):
    r = subprocess.run([sys.executable, "-c", "import mujoco,numpy as np; m=mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size=\"1\"/></worldbody></mujoco>'); d=mujoco.MjData(m); r=mujoco.Renderer(m,64,64); r.update_scene(d); print(r.render().mean())"], env={**os.environ, "MUJOCO_GL": backend}, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-200:]
for b in ("egl", "osmesa"):
    ok, msg = try_gl(b); print(b, "OK" if ok else "FAIL", msg if not ok else "")
    if ok: os.environ["MUJOCO_GL"] = b; break
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

In [ ]:
!python sim/vla-bed/gpu/preflight.py --execute --dataset-root datasets/vla-bed/v2 --min-disk-gb 10 | tail -c 1500

In [ ]:
# Three 10-step timings (each writes run_record.json with steps/s): bfloat16 as shipped, float16 VLM, and a smaller batch.
import subprocess, json, pathlib, sys
TRIALS = [("bf16-b32", ["--batch-size", "32"]), ("fp16-b32", ["--batch-size", "32", "--vlm-dtype", "float16"]), ("bf16-b16", ["--batch-size", "16"])]
records = {}
for tag, extra in TRIALS:
    out = pathlib.Path(f"/kaggle/working/smoke/{tag}")
    cmd = [sys.executable, "sim/vla-bed/gpu/train.py", "--run", "baseline", "--mode", "smoke", "--steps", "10", "--save-freq", "10", "--output-root", str(out), "--execute", *extra]
    print("\n===", tag, " ".join(cmd[2:]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-600:]); print(r.stderr[-800:] if r.returncode else "")
    rec = out / "baseline" / "smoke" / "run_record.json"
    records[tag] = json.loads(rec.read_text()) if rec.exists() else {"status": f"failed rc={r.returncode}"}
    print(tag, {k: records[tag].get(k) for k in ("status", "steps_done", "steps_per_s", "peak_vram_gb", "wall_s")})
json.dump(records, open("/kaggle/working/smoke_records.json", "w"), indent=1)

In [ ]:
# The evaluator on this box: scripted control (renders + safety), then the 10-step checkpoint (GPU inference latency).
import subprocess, sys, glob, json, time
r0 = subprocess.run([sys.executable, "sim/vla-bed/evaluate.py", "--policy", "oracle", "--episodes", "5", "--label", "kaggle-oracle"], capture_output=True, text=True)
print(r0.stdout[-700:], r0.stderr[-600:] if r0.returncode else "")
ck = sorted(glob.glob("/kaggle/working/smoke/bf16-b32/baseline/smoke/checkpoints/*/pretrained_model"))
if ck:
    t0 = time.time()
    r = subprocess.run([sys.executable, "sim/vla-bed/evaluate.py", "--policy", "smolvla", "--run", "baseline", "--checkpoint", ck[-1], "--episodes", "2", "--label", "kaggle-smoke"], capture_output=True, text=True)
    print(r.stdout[-900:], r.stderr[-400:] if r.returncode else "")
    print("2 episodes wall", round(time.time() - t0, 1), "s")

In [ ]:
# Projection: hours for 5k / 10k / 20k steps and for evaluating 4 checkpoints x 100 episodes; what fits an 8 h session.
import json, glob
rec = json.load(open("/kaggle/working/smoke_records.json"))
evals = glob.glob("sim/vla-bed/results/p5/*/kaggle-smoke/nominal.json")
sec_per_chunk = None
if evals:
    e = json.load(open(evals[0])); sec_per_chunk = e["kaggle-smoke"]["latency_s_mean"]
eval_h = (100 * 4 * ((sec_per_chunk or 1.0) * 4 + 1.5)) / 3600   # 4 checkpoints, ~4 chunks + ~1.5 s physics per episode
print(f"eval latency {sec_per_chunk} s/chunk → 4 checkpoints x 100 episodes ≈ {eval_h:.2f} h")
print(f"{'trial':10} {'steps/s':>8} {'VRAM GB':>8} {'5k h':>6} {'10k h':>6} {'20k h':>6}  fits 8 h?")
best = None
for tag, r in rec.items():
    sps = r.get("steps_per_s")
    if not sps: print(f"{tag:10} {'—':>8}  {r.get('status')}"); continue
    h = {n: n / sps / 3600 for n in (5000, 10000, 20000)}
    fit = "10k" if h[10000] + eval_h <= 8 else ("5k" if h[5000] + eval_h <= 8 else "no")
    print(f"{tag:10} {sps:8.3f} {r.get('peak_vram_gb') or 0:8.1f} {h[5000]:6.2f} {h[10000]:6.2f} {h[20000]:6.2f}  {fit}")
    if fit != "no" and (best is None or sps > best[1]): best = (tag, sps, fit)
print("\nDECISION:", f"Route K with {best[0]} ({best[2]} steps)" if best else "nothing fits 8 h → RunPod (GPU-GATE.md)")
json.dump({"records": rec, "sec_per_chunk": sec_per_chunk, "eval_hours_4x100": eval_h, "best": best}, open("/kaggle/working/projection.json", "w"), indent=1)

Paste the table above (or `/kaggle/working/projection.json`) back into the session. The decision rule is in `sim/vla-bed/GPU-GATE.md`, Route K.